# Recycling Bin Classifier: Smart Waste Sorting with Image Transforms

A smart recycling facility needs a computer vision system to identify waste categories from photographs. Images arrive at varying sizes and orientations, so every image must pass through a carefully designed preprocessing and augmentation pipeline before a model can use it.

## Tasks

1. Build `train_transform`: a `transforms.Compose` pipeline that resizes each image to 64x64, applies at least one geometric augmentation, applies at least one colour augmentation (`ColorJitter`), converts to a float32 tensor, normalises with ImageNet mean and std, and applies `RandomErasing` after the tensor is formed.

2. Build `test_transform`: a `transforms.Compose` pipeline that resizes each image to 64x64, converts to a float32 tensor, and normalises with the same ImageNet mean and std. No augmentation steps.

3. Attach `train_transform` to a `torchvision.datasets.ImageFolder` dataset pointed at `images/garbage_classification/` and store it in `train_data`. Attach `test_transform` to a second `ImageFolder` over the same folder and store it in `test_data`.

4. Fetch one sample from each dataset. Confirm the output tensor has shape `[3, 64, 64]` and dtype `torch.float32`. Store tensors in `sample_tensor_train` and `sample_tensor_test`.

5. Demonstrate that `train_transform` is stochastic and `test_transform` is deterministic by applying each to index 0 twice and comparing the results.

**Required variable names:** `train_transform`, `test_transform`, `train_data`, `test_data`, `sample_tensor_train`, `sample_tensor_test`

In [ ]:
# Run this cell before writing your solution
import warnings
warnings.filterwarnings('ignore')

import zipfile
import os
import random
import numpy as np
import torch
import torchvision
from torchvision import transforms, datasets
from torchvision.transforms import (
    Compose, Resize, ToTensor, Normalize, RandomErasing,
    RandomHorizontalFlip, RandomVerticalFlip, RandomCrop,
    RandomResizedCrop, RandomRotation, RandomAffine,
    ColorJitter, RandomGrayscale, GaussianBlur
)
from PIL import Image

# Extract the student archive
zipfile.ZipFile("images_TRANSF.zip").extractall("images")

DATA_DIR = "images/garbage_classification"
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print("Classes found:", sorted(os.listdir(DATA_DIR)))

In [ ]:
# Reference solution
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

IMG_SIZE = 64

# 1. Build train_transform
train_transform = Compose([
    Resize((IMG_SIZE + 16, IMG_SIZE + 16)),         # resize to slightly larger than target
    RandomHorizontalFlip(p=0.5),                    # geometric augmentation
    RandomRotation(degrees=15),                     # geometric augmentation
    ColorJitter(brightness=0.3, contrast=0.3,
                saturation=0.2, hue=0.1),           # colour augmentation
    RandomCrop(IMG_SIZE),                           # geometric augmentation -> final 64x64
    ToTensor(),                                     # PIL Image -> float32 tensor [0,1]
    Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),# normalise with ImageNet stats
    RandomErasing(p=0.5),                           # tensor-level augmentation
])

# 2. Build test_transform (no augmentation, deterministic)
test_transform = Compose([
    Resize((IMG_SIZE, IMG_SIZE)),                   # resize to 64x64
    ToTensor(),                                     # PIL Image -> float32 tensor
    Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),# normalise with same ImageNet stats
])

# 3. Attach transforms to ImageFolder datasets
train_data = datasets.ImageFolder(DATA_DIR, transform=train_transform)
test_data  = datasets.ImageFolder(DATA_DIR, transform=test_transform)

print(f"train_data: {len(train_data)} images, classes: {train_data.classes}")
print(f"test_data:  {len(test_data)} images")

# 4. Fetch one sample from each dataset
sample_tensor_train, train_label = train_data[0]
sample_tensor_test,  test_label  = test_data[0]

print(f"\nsample_tensor_train: shape={sample_tensor_train.shape}, dtype={sample_tensor_train.dtype}")
print(f"sample_tensor_test:  shape={sample_tensor_test.shape},  dtype={sample_tensor_test.dtype}")

assert sample_tensor_train.shape == torch.Size([3, 64, 64]), "Expected shape [3, 64, 64]"
assert sample_tensor_train.dtype == torch.float32, "Expected float32"
assert sample_tensor_test.shape  == torch.Size([3, 64, 64]), "Expected shape [3, 64, 64]"
assert sample_tensor_test.dtype  == torch.float32, "Expected float32"

# 5. Demonstrate stochastic vs deterministic behaviour
img_path, _ = train_data.samples[0]
pil_img = Image.open(img_path).convert("RGB")

# train_transform is stochastic
out_train_1 = train_transform(pil_img)
out_train_2 = train_transform(pil_img)
print(f"\ntrain_transform outputs differ (stochastic): {not torch.equal(out_train_1, out_train_2)}")

# test_transform is deterministic
out_test_1  = test_transform(pil_img)
out_test_2  = test_transform(pil_img)
print(f"test_transform outputs identical (deterministic): {torch.equal(out_test_1, out_test_2)}")